# Recursive Language Models on Lunar Sandbox

Implementation of the [Recursive Language Models](https://arxiv.org/abs/2512.24601) (Zhang, Kraska & Khattab 2025) pattern using the Lunar Sandbox SDK.

**Core idea:** Instead of stuffing a huge prompt into the context window, the LLM gets a REPL (our sandbox) and a `llm_query` tool to recursively call itself on sub-problems. The agent decomposes the input, processes chunks via recursive self-calls, and aggregates results — all inside an isolated container.

```
   Root LLM call
   ├── examines long input
   ├── writes code to chunk it
   ├── llm_query(chunk_1) → sub-LLM call
   ├── llm_query(chunk_2) → sub-LLM call
   ├── llm_query(chunk_3) → sub-LLM call
   └── aggregates results → final answer
```

All steps are auto-traced to the dashboard at http://localhost:3000.

In [1]:
# Install dependencies (run once)
!pip install openai -q

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

# Ensure docker CLI is on PATH (Jupyter kernels may not inherit the full shell PATH)
os.environ["PATH"] = "/usr/local/bin:/opt/homebrew/bin:" + os.environ.get("PATH", "")

# Set your API key (Mistral, OpenAI, or any OpenAI-compatible provider)
os.environ["MISTRAL_API_KEY"] = ""  # <-- set your key

## The RLM Engine

We extend the sandbox tool loop with a `llm_query` tool, letting the agent recursively call itself on sub-problems. This is the key primitive from the RLM paper — the model can decompose long inputs and self-delegate.

In [3]:
import json
from openai import OpenAI
from lunar_sandbox import Session


RLM_SYSTEM_PROMPT = """You are a Recursive Language Model. You solve tasks by decomposing them \
into smaller sub-problems, using code execution in a sandbox and recursive LLM calls.

You have these tools:
- run_command: Execute shell commands in the sandbox (install packages, run scripts, etc.)
- write_file: Write files to /workspace
- read_file: Read files from /workspace
- list_files: List directory contents
- llm_query: Recursively call yourself on a sub-problem. Use this to break complex tasks \
into smaller pieces. The sub-call gets its own context and returns a text answer.

Strategy:
1. If the task is simple enough, solve it directly with code.
2. If the task involves long input or multiple parts, decompose it and use llm_query \
for each sub-part.
3. Aggregate sub-results into a final answer.

Always prefer decomposition over trying to solve everything at once."""


# The extra llm_query tool definition (added alongside sandbox tools)
LLM_QUERY_TOOL = {
    "type": "function",
    "function": {
        "name": "llm_query",
        "description": (
            "Recursively call the LLM on a sub-problem. Use this to decompose "
            "complex tasks: pass a focused sub-question and optional context. "
            "Returns the LLM's text answer for that sub-problem."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The sub-question or sub-task to solve.",
                },
                "context": {
                    "type": "string",
                    "description": "Optional context data (e.g. a text chunk to analyze).",
                },
            },
            "required": ["query"],
        },
    },
}


def rlm_completion(
    client: OpenAI,
    model: str,
    session: Session,
    task: str,
    *,
    max_iterations: int = 30,
    depth: int = 0,
    max_depth: int = 2,
    verbose: bool = True,
) -> str:
    """Run one RLM completion: the agent can use sandbox tools + llm_query."""
    indent = "  " * depth

    # Sandbox tools + the recursive llm_query tool
    tools = session.tools(format="openai") + [LLM_QUERY_TOOL]

    messages = [
        {"role": "system", "content": RLM_SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]

    for step in range(max_iterations):
        resp = client.chat.completions.create(
            model=model, messages=messages, tools=tools,
        )
        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            if verbose:
                print(f"{indent}[depth={depth}] Done after {step} steps")
            return msg.content or ""

        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)

            if name == "llm_query":
                # Recursive self-call
                sub_query = args["query"]
                sub_context = args.get("context", "")
                sub_prompt = sub_query
                if sub_context:
                    sub_prompt += f"\n\nContext:\n{sub_context}"

                if verbose:
                    preview = sub_query[:80] + ("..." if len(sub_query) > 80 else "")
                    print(f"{indent}[depth={depth}] llm_query -> {preview}")

                if depth < max_depth:
                    result = rlm_completion(
                        client, model, session, sub_prompt,
                        max_iterations=max_iterations,
                        depth=depth + 1,
                        max_depth=max_depth,
                        verbose=verbose,
                    )
                else:
                    # At max depth, do a single non-recursive call
                    sub_resp = client.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": sub_prompt}],
                    )
                    result = sub_resp.choices[0].message.content or ""
            else:
                # Sandbox tool call
                result = session.call_tool(name, args)
                if verbose:
                    preview = result[:120].replace("\n", " ")
                    print(f"{indent}[depth={depth}] {name} -> {preview}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result[:8000],
            })

    return messages[-1].get("content", "") if isinstance(messages[-1], dict) else ""


print("RLM engine ready.")

2026-03-22 15:56:20 [debug    ] seccomp_module_not_available  
RLM engine ready.


## Example 1 — Analyze a long document by decomposition

The agent gets a long multi-section document. Instead of processing it all at once, it writes the text to the sandbox, chunks it, and uses `llm_query` to analyze each section recursively.

In [4]:
client = OpenAI(
    api_key=os.environ["MISTRAL_API_KEY"],
    base_url="https://api.mistral.ai/v1",
)

LONG_DOCUMENT = """
=== SECTION 1: Q1 Revenue ===
Total revenue for Q1 was $12.4M, up 18% YoY. SaaS subscriptions contributed $8.1M
while professional services accounted for $4.3M. Gross margin improved to 72% from
68% in the prior year. Customer acquisition cost decreased by 12% due to improved
organic marketing channels. Net new ARR was $2.1M with 340 new customers added.
Enterprise segment grew 24% while SMB grew 11%. Churn rate held steady at 4.2%.

=== SECTION 2: Q2 Revenue ===
Q2 revenue reached $14.7M, a 22% YoY increase. SaaS subscriptions grew to $10.2M
driven by upsells in the enterprise segment. Professional services remained flat at
$4.5M. Gross margin reached 74%. A new partnership with AWS Marketplace contributed
$1.8M in net new pipeline. Net new ARR was $3.4M with 410 new customers. Enterprise
grew 31% while SMB grew 8%. Churn rate improved to 3.8%.

=== SECTION 3: Q3 Revenue ===
Q3 saw revenue of $16.1M, up 26% YoY. SaaS subscriptions hit $11.8M as the new
premium tier launched in August drove significant upgrades. Professional services
dipped to $4.3M due to seasonal effects. Gross margin was 73%. The company signed
its largest deal to date at $1.2M ARR with a Fortune 500 company. Net new ARR was
$4.1M with 380 new customers. Enterprise grew 35% while SMB grew 6%. Churn dropped
to 3.5%.

=== SECTION 4: Q4 Revenue ===
Q4 revenue was $19.3M, up 31% YoY and 20% QoQ. SaaS subscriptions reached $14.1M.
Professional services recovered to $5.2M with end-of-year implementation projects.
Gross margin hit a record 76%. Three Fortune 500 deals closed totaling $3.8M ARR.
Net new ARR was $5.7M with 520 new customers, the best quarter in company history.
Enterprise grew 42% while SMB grew 4%. Churn rate was 3.1%, the lowest ever.

=== SECTION 5: Annual Outlook ===
Full-year revenue totaled $62.5M, up 24% YoY. The board approved a 2026 target of
$85M (36% growth). Key investments planned: international expansion into EMEA ($4M
budget), new AI-powered analytics module ($6M R&D), and doubling the enterprise
sales team from 12 to 24 reps. The company expects to reach cash-flow breakeven
by Q3 2026.
"""

with Session("rlm-document-analysis", image="python:3.12-slim") as s:
    answer = rlm_completion(
        client,
        model="mistral-small-latest",
        session=s,
        task=(
            "Analyze the following financial report. For each quarterly section, "
            "extract the key metrics (revenue, growth, margins, churn). Then "
            "synthesize an overall trend analysis with a final recommendation.\n\n"
            "Use llm_query to analyze each section independently, then aggregate.\n\n"
            f"Document:\n{LONG_DOCUMENT}"
        ),
        max_iterations=25,
        max_depth=2,
    )
    print("\n" + "=" * 60)
    print("FINAL ANALYSIS:")
    print("=" * 60)
    print(answer)
    s.finish(score=1.0)

2026-03-22 15:56:26 [debug    ] trajectory_store_opened        db_path=/Users/diogovieira/Developer/sandbox/trajectories/trajectories.db
2026-03-22 15:56:26 [info     ] episode_ingested               episode_id=ep-43c00e9d0785 step_count=0
2026-03-22 15:56:26 [debug    ] docker_sandbox_initialized     sandbox_id=session-2174363ff99f state=created
2026-03-22 15:56:26 [info     ] docker_sandbox_creating        sandbox_id=session-2174363ff99f
2026-03-22 15:56:28 [debug    ] health_mount_recorded          mount_count=1 sandbox_id=session-2174363ff99f
2026-03-22 15:56:28 [info     ] docker_sandbox_created         container_id=b257c9bd9afa image=python:3.12-slim sandbox_id=session-2174363ff99f
2026-03-22 15:56:28 [info     ] session_started                image=python:3.12-slim session_episode=ep-43c00e9d0785 session_sandbox=session-2174363ff99f task=rlm-document-analysis url=http://localhost:3000/runs/ep-43c00e9d0785
Session started: http://localhost:3000/runs/ep-43c00e9d0785
[depth=0] llm_

## Example 2 — Recursive code generation

The agent builds a multi-module project by recursively delegating each module to a sub-call, then integrating and testing everything in the sandbox.

In [ ]:
with Session("rlm-recursive-codegen", image="python:3.12-slim") as s:
    s.run("pip install -q pytest")

    answer = rlm_completion(
        client,
        model="mistral-small-latest",
        session=s,
        task=(
            "Build a Python statistics library with 3 modules. "
            "Use llm_query to design each module independently:\n"
            "1. stats/descriptive.py — mean, median, mode, std_dev, variance\n"
            "2. stats/correlation.py — pearson, spearman rank correlation\n"
            "3. stats/regression.py — simple linear regression (slope, intercept, r_squared)\n\n"
            "For each module, use llm_query to get the implementation, then "
            "write it to the sandbox. After all modules are written, create "
            "tests/test_stats.py with comprehensive pytest tests and run them.\n\n"
            "No external dependencies — pure Python only."
        ),
        max_iterations=30,
        max_depth=2,
    )
    print("\n" + "=" * 60)
    print("RESULT:")
    print("=" * 60)
    print(answer)

    # Show what was built
    print("\n=== Files ===")
    print(s.run("find /workspace -name '*.py' | head -20"))
    print("\n=== Test results ===")
    print(s.run("cd /workspace && python -m pytest tests/ -v"))

    s.finish(score=1.0)

## How it works

The RLM pattern maps directly to Lunar Sandbox:

| RLM Paper | Lunar Sandbox |
|-----------|---------------|
| Persistent REPL environment | Docker sandbox (run_command, write_file, etc.) |
| `llm_query()` primitive | Custom `llm_query` tool → recursive `rlm_completion()` call |
| Context as variables | Files in `/workspace` |
| Execution trajectory | Auto-traced to dashboard via `Session` |

The `rlm_completion` function is ~60 lines. It runs the standard tool loop but intercepts `llm_query` calls to recurse into itself with `depth + 1`, up to `max_depth`.